# 🃏 Poker Arena — Servidor de Visão (Qwen3-VL) no Colab Pro

**O quê:** este notebook transforma a GPU do Colab num **leitor de telas de poker**. Ele sobe o **Qwen3-VL-4B** (um *Vision-Language Model*) como servidor e abre um **túnel público** — sua máquina local fala com ele por uma URL só.

**Por que isso resolve o problema das 1000 UIs:** o detector YOLO (F2) é *keyed a pixels* e não lê layouts nunca vistos (leu 0/2 cartas no PokerTH real). Um VLM lê carta e número **semanticamente** → uma UI inédita já é "in-distribution". É a peça agnóstica de verdade.

## Arquitetura (a GPU está no Colab, o backend em casa)

```
[SUA MÁQUINA]  browser captura a tela → backend FastAPI (local)
                                              │  POST da imagem
                                              ▼  (via túnel público)
[COLAB PRO]    cloudflared  →  llama-server (Qwen3-VL na GPU)  →  JSON do estado
```

O backend já está pronto pra isso: o `vlm_reader.py` fala com qualquer endpoint OpenAI-compatível via `POKER_VLM_URL`. Você só cola a URL do túnel.

## ⚠️ O caveat honesto do ≤4s

Com a GPU no Colab, a latência = **upload da imagem (local→Colab) + inferência**. Isso **aperta o orçamento de 4s** da banca. Por isso o passo final é rodar `scripts/vlm_eval.py` **no seu backend local** apontando pro túnel — ele mede a latência **real, ponta a ponta**. Não prometa ≤4s à banca antes de ver esse número. Lembre: o VLM só dispara no **abstain** (UI inédita); o comum é lido em milissegundos pelo caminho rápido local.

## Como usar

1. **Runtime → GPU** (T4/L4/A100) e rode a célula abaixo.
2. Espere subir (compila o llama.cpp ~8-10 min na 1ª vez, baixa ~3 GB do modelo).
3. **Copie** a linha `$env:POKER_VLM_URL = "..."` que ele imprime → cole no seu PowerShell local.
4. **Prova na hora:** suba um screenshot de mesa (ex.: o do PokerTH) no passo 6 e veja o Qwen3-VL ler.
5. **Meça de verdade** local: `uv run python scripts/vlm_eval.py scripts/real_eval/imgs`.
6. Deixe a célula rodando durante a demo (ela mantém o túnel vivo).

> Notebook em **célula única** (1 markdown + 1 code). Se a visão falhar por falta do `mmproj`, veja `server.log` e rode de novo — o `-hf` costuma baixar o projetor de visão junto.


In [ ]:
# ================================================================
#  Poker Arena — Servidor de VISAO (Qwen3-VL) na GPU do Colab Pro
# ================================================================
# Este notebook da OLHOS pro seu copiloto: sobe o Qwen3-VL (um VLM que le
# QUALQUER mesa de poker, mesmo uma UI nunca vista) como um servidor na GPU
# do Colab, e abre um tunel publico pra sua maquina LOCAL falar com ele.
# Voce copia UMA linha pro backend e o "F3-VLM" ganha vida. Respira: eu
# explico e mostro cada passo.

import base64, json, os, re, subprocess, time, urllib.request

print("🃏  Vamos dar OLHOS pro copiloto. Sem susto — eu narro cada passo.\n")

# --- Passo 0: cade a GPU? --------------------------------------------------
print("👀  Passo 0 — vendo que GPU o Colab te deu...")
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
if not gpu:
    raise SystemExit("🛑  SEM GPU. Menu: Ambiente de execucao → Alterar tipo → GPU (T4/L4/A100), e rode de novo.")
print(f"✅  Achei: {gpu}  — e nela que o Qwen3-VL vai rodar.\n")

# --- Passo 1: ferramentas (libcurl p/ baixar o modelo, cloudflared p/ tunel)
print("🔧  Passo 1 — instalando o basico (libcurl + cloudflared)... (~30s)")
subprocess.run(["apt-get", "-qq", "install", "-y", "libcurl4-openssl-dev"], capture_output=True)
subprocess.run(["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/"
                "download/cloudflared-linux-amd64", "-O", "cloudflared"], check=True)
subprocess.run(["chmod", "+x", "cloudflared"], check=True)
print("✅  Ferramentas prontas.\n")

# --- Passo 2: compilar o llama.cpp com CUDA --------------------------------
# Por que compilar? O Qwen3-VL e novo (out/2025); a gente quer o llama.cpp
# fresquinho que ja sabe ler esse modelo. Leva ~8-10 min UMA vez so.
if not os.path.exists("llama.cpp/build/bin/llama-server"):
    print("🏗️   Passo 2 — compilando o llama.cpp com CUDA (~8-10 min, pega um cafe ☕)...")
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/ggml-org/llama.cpp"], check=True)
    subprocess.run(["cmake", "-S", "llama.cpp", "-B", "llama.cpp/build",
                    "-DGGML_CUDA=ON", "-DLLAMA_CURL=ON"], check=True)
    subprocess.run(["cmake", "--build", "llama.cpp/build", "--config", "Release",
                    "-j", "--target", "llama-server"], check=True)
    print("✅  llama-server compilado.\n")
else:
    print("✅  Passo 2 — llama-server ja estava compilado (pulei).\n")

# --- Passo 3: subir o Qwen3-VL como servidor -------------------------------
print("🚀  Passo 3 — subindo o Qwen3-VL-4B-Instruct (baixa ~3 GB na 1a vez)...")
log = open("server.log", "w")
server = subprocess.Popen(
    ["llama.cpp/build/bin/llama-server",
     "-hf", "unsloth/Qwen3-VL-4B-Instruct-GGUF:Q4_K_M",  # baixa modelo + mmproj (visao)
     "-ngl", "99", "--host", "0.0.0.0", "--port", "8080", "--jinja", "-c", "8192"],
    stdout=log, stderr=subprocess.STDOUT)


def _wait_health(url, timeout=900):
    t0 = time.time()
    while time.time() - t0 < timeout:
        if server.poll() is not None:
            print("🛑  O servidor morreu. Fim do log:\n", open("server.log").read()[-2500:])
            raise SystemExit
        try:
            urllib.request.urlopen(url, timeout=3)
            return True
        except Exception:
            time.sleep(4)
    return False


print("⏳  Esperando carregar (baixa o modelo + sobe na GPU — pode levar minutos na 1a vez)...")
if not _wait_health("http://localhost:8080/health"):
    raise SystemExit("🛑  Nao ficou pronto a tempo. Rode a celula de novo (o download continua do cache).")
print("✅  Qwen3-VL no ar em http://localhost:8080\n")

# --- Passo 4: abrir o tunel publico ----------------------------------------
print("🌍  Passo 4 — abrindo um tunel publico (cloudflared) pra sua maquina alcancar...")
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8080",
                           "--no-autoupdate"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
public_url = None
t0 = time.time()
while time.time() - t0 < 60:
    line = tunnel.stdout.readline()
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line or "")
    if m:
        public_url = m.group(0)
        break
if not public_url:
    raise SystemExit("🛑  Nao peguei a URL do tunel. Rode a celula de novo.")
vlm_url = public_url + "/v1/chat/completions"
print("✅  Tunel aberto!\n")

# --- Passo 5: a linha que voce cola no backend LOCAL -----------------------
print("=" * 66)
print("📋  COPIE e rode no seu PowerShell LOCAL (dentro da pasta backend):\n")
print(f'    $env:POKER_VLM_URL = "{vlm_url}"\n')
print("    E entao MEÇA de verdade (leitura + latencia vs 4s):")
print("    uv run python scripts/vlm_eval.py scripts/real_eval/imgs")
print("=" * 66 + "\n")

# --- Passo 6: PROVA aqui mesmo — o VLM le um screenshot? -------------------
print("🧪  Passo 6 — teste NA HORA: suba 1 screenshot de mesa de poker (ex.: o do")
print("    PokerTH que o F2 reprovava). Cancele o upload pra pular.")
try:
    from google.colab import files
    up = files.upload()
except Exception:
    up = {}
if up:
    name = list(up)[0]
    b64 = base64.b64encode(up[name]).decode()
    body = {"model": "qwen3-vl", "temperature": 0, "max_tokens": 256,
            "response_format": {"type": "json_object"},
            "messages": [{"role": "user", "content": [
                {"type": "text", "text":
                 "Leia esta mesa de poker Texas Hold'em e devolva SOMENTE JSON: "
                 '{"hole":[cartas do heroi, embaixo], "board":[cartas do centro], '
                 '"pot":inteiro ou null}. Notacao: rank[2 3 4 5 6 7 8 9 T J Q K A] + '
                 'naipe[s h d c], ex "As","Td","6h". Use T para 10. Nao invente cartas.'},
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}}]}]}
    t0 = time.time()
    req = urllib.request.Request("http://localhost:8080/v1/chat/completions",
                                 data=json.dumps(body).encode(),
                                 headers={"Content-Type": "application/json"})
    resp = json.loads(urllib.request.urlopen(req, timeout=90).read())
    dt = time.time() - t0
    print(f"\n🤖  O Qwen3-VL leu (inferencia na GPU em {dt:.1f}s):")
    print("   ", resp["choices"][0]["message"]["content"])
    print("\n👉  Confere com a tela: acertou as cartas? ESSE e o teste que o F2 reprovava.")
    print("    (a latencia daqui e SO a inferencia; do seu backend soma o upload local→Colab.)")
else:
    print("   (pulou — sem imagem. Da pra testar do backend local com vlm_eval.py.)\n")

# --- Passo 7: manter vivo --------------------------------------------------
print("\n💓  Passo 7 — deixando o servidor VIVO. NAO feche esta aba durante a demo.")
print("    Enquanto esta celula roda, sua URL funciona; ao parar, o tunel fecha.\n")
try:
    while True:
        time.sleep(120)
        estado = "ok" if server.poll() is None else "CAIU — rode a celula de novo"
        print(f"    💓 {time.strftime('%H:%M:%S')} — servidor {estado} | {public_url}")
except KeyboardInterrupt:
    print("👋  Encerrando. Tunel fechado.")
